# Titanic Survival Classification — EDA and Model Pipeline

This notebook explores the Kaggle Titanic training data, documents data quality and visual patterns, and launches the reproducible training/evaluation pipeline. Run from the repository root.

**Dataset requirement:** place Kaggle `train.csv` at `data/raw/train.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('../data/raw/train.csv') if not Path('data/raw/train.csv').exists() else Path('data/raw/train.csv')
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())

## 1. Inspect structure and data quality

In [ ]:
display(df.info())
display(df.describe(include='all').T)
missing = df.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({'missing_count': missing, 'missing_percent': (missing / len(df) * 100).round(2)}))
display(df['Survived'].value_counts().rename('count').to_frame())

## 2. Distributions and categorical value counts

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)
for col in ['Pclass', 'Sex', 'Embarked']:
    if col in df.columns:
        print(f'\\n{col} value counts:')
        display(df[col].value_counts(dropna=False).to_frame('count'))
df[numeric_cols].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()

## 3. Visual EDA — box plot, scatter plot and pair plot

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='Survived', y='Age')
plt.title('Age distribution by survival label')
plt.show()

plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x='Age', y='Fare', hue='Survived', alpha=0.7)
plt.title('Age vs Fare by survival label')
plt.show()

pair_cols = [c for c in ['Survived', 'Pclass', 'Age', 'Fare', 'SibSp', 'Parch'] if c in df.columns]
sns.pairplot(df[pair_cols].dropna().sample(min(250, len(df.dropna(subset=pair_cols))), random_state=42), hue='Survived')
plt.show()

## 4. Correlation heatmap and observations

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Numeric feature correlation heatmap')
plt.show()

print('Observation prompts — inspect your own charts and write evidence-based findings:')
print('1. Compare survival proportions across passenger classes.')
print('2. Compare survival proportions across sex categories.')
print('3. Inspect missingness in Age, Cabin and Embarked and consider its impact.')
display(pd.crosstab(df['Sex'], df['Survived'], normalize='index').round(3))
display(pd.crosstab(df['Pclass'], df['Survived'], normalize='index').round(3))

## 5. Feature engineering preview

In [ ]:
from src.features import add_features
engineered = add_features(df)
display(engineered[['Name', 'Title', 'SibSp', 'Parch', 'FamilySize', 'IsAlone']].head())

## 6. Train, compare and evaluate models

The training script performs train/test splitting, pipeline-based imputation/encoding/scaling, Logistic Regression, Decision Tree, Random Forest, 5-fold CV, GridSearchCV, evaluation, ROC plots and joblib saving. Run it from the repository root.

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'src.train'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Training script failed. See error output above.')

## 7. Inspect generated metrics and report

In [ ]:
metrics_path = Path('reports/model_metrics.csv')
if not metrics_path.exists():
    metrics_path = Path('../reports/model_metrics.csv')
display(pd.read_csv(metrics_path))
print((metrics_path.parent / 'EVALUATION_REPORT.md').read_text(encoding='utf-8'))